<a href="https://colab.research.google.com/github/seeuny2022-commits/finflag/blob/main/Seeun_PARK_FinFlag.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [9]:
!pip -q install google-genai datasets gradio yfinance

Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/pip/_internal/cli/base_command.py", line 109, in _run_wrapper
    status = _inner_run()
  File "/usr/local/lib/python3.13/dist-packages/pip/_internal/cli/base_command.py", line 102, in _inner_run
    return self.run(options, args)
           ~~~~~~~~^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.13/dist-packages/pip/_internal/cli/req_command.py", line 106, in wrapper
    return func(self, options, args)
  File "/usr/local/lib/python3.13/dist-packages/pip/_internal/commands/install.py", line 541, in run
    conflicts = self._determine_conflicts(to_install)
  File "/usr/local/lib/python3.13/dist-packages/pip/_internal/commands/install.py", line 662, in _determine_conflicts
    return check_install_conflicts(to_install)
  File "/usr/local/lib/python3.13/dist-packages/pip/_internal/operations/check.py", line 110, in check_install_conflicts
    package_set, _ = create_package_set_from_installed()
            

In [10]:
import json, time
import pandas as pd
from google import genai
from google.genai import types
from google.colab import userdata, files

# Read the secret key (never paste the key itself here, because this code goes on GitHub)
client = genai.Client(api_key=userdata.get("GEMINI_API_KEY"))

# Find the "flash" models your key can use (fast + free)
names = [m.name.replace("models/", "") for m in client.models.list()]
flash = [n for n in names if "flash" in n and not any(x in n for x in ["image", "tts", "live", "audio", "embed", "lite"])]
print("Flash models available:", flash)

MODEL = flash[0]
print("Using:", MODEL)

Flash models available: ['gemini-2.5-flash', 'gemini-3.5-flash', 'gemini-omni-flash-preview', 'gemini-3.6-flash', 'gemini-3.7-flash', 'gemini-3.8-flash']
Using: gemini-2.5-flash


In [6]:
from datasets import load_dataset

ds = load_dataset("gtfintechlab/VideoConviction")
print(ds)

df = ds[list(ds.keys())[0]].to_pandas()   # turn it into a table
print(df.shape)                            # (rows, columns)
print(df.columns.tolist())                 # the column names
df.head(3)

README.md:   0%|          | 0.00/4.47k [00:00<?, ?B/s]

VideoConviction.csv: reconstructing file:   0%|          |  0.00B / 15.4MB            

VideoConviction.csv: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/760 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['id', 'derived_inner_id', 'video_id', 'start', 'end', 'action', 'action_source', 'conviction_score', 'ticker_name', 'action_date', 'price', 'quantity', 'video_title', 'annotation_id', 'is_rec_present', 'original_inner_id', 'original_video_title', 'publishedAt', 'channelId', 'channelTitle', 'videoDescription', 'tags', 'defaultAudioLanguage', 'duration', 'isCaptionAvailable', 'viewCount', 'likeCount', 'favoriteCount', 'commentCount', 'comments', 'channelDescription', 'channelViewCount', 'channelSubscriberCount', 'videoCount', 'channelCategory', 'transcript', 'youtube_video_url', 'segment_transcript'],
        num_rows: 760
    })
})
(760, 38)
['id', 'derived_inner_id', 'video_id', 'start', 'end', 'action', 'action_source', 'conviction_score', 'ticker_name', 'action_date', 'price', 'quantity', 'video_title', 'annotation_id', 'is_rec_present', 'original_inner_id', 'original_video_title', 'publishedAt', 'channelId', 'channelTitle', 'vide

,id,derived_inner_id,video_id,start,end,action,action_source,conviction_score,ticker_name,action_date,...,commentCount,comments,channelDescription,channelViewCount,channelSubscriberCount,videoCount,channelCategory,transcript,youtube_video_url,segment_transcript
0,125899618,1,0CJU8R4oNFk,109.871177,158.538947,Buy,Title,2.0,VERI,None,...,168.0,"["".I don’t who, but someone actually needs to ...",Welcome to your chance to create the financial...,43592318,641000,1168,Category 1,"Hey Bowtie Nation, Joseph Hogue here with the...",https://www.youtube.com/watch?v=0CJU8R4oNFk,these in a minute. First up is $225 million d...
1,125899618,1,0CJU8R4oNFk,197.798669,267.774387,Buy,Selected region,2.0,ZYXI,None,...,168.0,"["".I don’t who, but someone actually needs to ...",Welcome to your chance to create the financial...,43592318,641000,1168,Category 1,"Hey Bowtie Nation, Joseph Hogue here with the...",https://www.youtube.com/watch?v=0CJU8R4oNFk,"up and make your first deposit. Zynex, ticker..."
2,125899618,1,0CJU8R4oNFk,465.293375,514.698535,Buy,Title,2.0,MITK,None,...,168.0,"["".I don’t who, but someone actually needs to ...",Welcome to your chance to create the financial...,43592318,641000,1168,Category 1,"Hey Bowtie Nation, Joseph Hogue here with the...",https://www.youtube.com/watch?v=0CJU8R4oNFk,"Our next stock, $500 million dollar Midic Sys..."


In [7]:
TEXT_COL = "segment_transcript" if "segment_transcript" in df.columns else "transcript"
VIDEO_COL = "youtube_video_url"
CHANNEL_COL = next((c for c in df.columns if "channel" in c.lower()), None)
print("text:", TEXT_COL, "| video:", VIDEO_COL, "| channel:", CHANNEL_COL)

# Remove empty or very short segments
df = df[df[TEXT_COL].fillna("").str.len() > 80].reset_index(drop=True)
df["seg_id"] = range(len(df))
print(len(df), "usable segments from", df[VIDEO_COL].nunique(), "videos")

print("\nExample segment:\n", df.loc[0, TEXT_COL][:800])

text: segment_transcript | video: youtube_video_url | channel: channelId
678 usable segments from 214 videos

Example segment:
  these in a minute. First up is $225 million dollar Veritone Inc., ticker V-E-R-I, a cloud-based AI platform that structures audio and video data. And now Nation, if that sounds like a bunch of tech jargon, just understand that Veritone is in the convergence of what's going to be the three biggest trends over the next decade. Data analysis of audio and video content, cloud-based connectivity, and an AI platform that learns to become more effective. And the potential markets for these speak for themselves. The company's AIware platform is facing markets with double-digit annual growth and tens of billions in opportunities. Revenue has jumped 244% in the last two years and was even able to increase in that tough second quarter. Analysts have price targets ranging from $15 a share to as high 


In [8]:
FLAGS = ["GUARANTEE", "URGENCY", "NO_RISK", "CERTAINTY", "PROMOTION", "ALL_IN"]

# Shuffle, then keep only ONE segment per video (so the same video can't be in both dev and test)
one_per_video = df.sample(frac=1, random_state=42).drop_duplicates(VIDEO_COL)

# Pick 70: first 20 = dev (for improving the prompt), last 50 = test (locked until the end)
sample = one_per_video.sample(min(70, len(one_per_video)), random_state=42).reset_index(drop=True)
sample["split"] = ["dev"] * 20 + ["test"] * (len(sample) - 20)

sheet = sample[["seg_id", "split", VIDEO_COL, TEXT_COL]].copy()
sheet = sheet.rename(columns={TEXT_COL: "text", VIDEO_COL: "video"})
for f in FLAGS + ["RISKY"]:
    sheet[f] = ""          # empty columns for YOU to fill in

sheet.to_csv("labeling_sheet.csv", index=False)
files.download("labeling_sheet.csv")
print(sheet.split.value_counts())

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

split
test    50
dev     20
Name: count, dtype: int64
